In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # agent_obs_alert
# MAGIC
# MAGIC Replaces the comment-only `%sql` version, whose `-- condition: n > 0` comments were never
# MAGIC evaluated. This notebook:
# MAGIC 1. persists findings to `obs_incidents` (append-only, MERGE-deduped),
# MAGIC 2. evaluates each detector condition in Python,
# MAGIC 3. bounds every check in time,
# MAGIC 4. reports missing/stale upstream tables as `UNAVAILABLE` rather than crashing, so a broken
# MAGIC    detector is loud instead of silent under `If all done`,
# MAGIC 5. posts a triage-oriented Adaptive Card to Teams — once per incident, not once per run,
# MAGIC 6. raises when a detector is broken, so Databricks' own job notifications fire.
# MAGIC
# MAGIC **Required job parameters:** `env`, `lookback_minutes`. The Teams webhook is read from the
# MAGIC `obs-alerting` secrets scope (key `teams-webhook`), not a job parameter —
# MAGIC `webhook_configured` reflects whether that secret is set, not whether a job parameter was
# MAGIC wired correctly. Interactive and job runs both read the secret the same way.
# MAGIC
# MAGIC ## Severity semantics
# MAGIC `raise` fires **only on UNAVAILABLE**, so task status means *"is monitoring working"* rather
# MAGIC than *"did it find something"*. CRITICAL findings route to Teams and `obs_incidents`.
# MAGIC
# MAGIC The reason: `handover_delivery` is true and long-lived, so raising on CRITICAL made task 06
# MAGIC permanently red — which trains everyone to ignore it, the exact failure mode this notebook
# MAGIC exists to prevent.
# MAGIC
# MAGIC ## Notification policy
# MAGIC A CRITICAL incident notifies when first detected, then not again for 24 h unless still
# MAGIC unacknowledged. WARN findings are **not** notified — `shift_context_missing` and
# MAGIC `hallucination_signal_liveness` are standing conditions and would drown the channel. They
# MAGIC stay visible in task output. Broken detectors notify every run with no dedup.
# MAGIC
# MAGIC ## Thresholds
# MAGIC Recorded with their basis in `mq_gmdf_dev.oil_obs.threshold_basis`. Every one is provisional:
# MAGIC there are no labelled examples yet, so these are anchored to observed ranges rather than to
# MAGIC confirmed incidents. `SELECT * FROM threshold_basis WHERE status LIKE 'unvalidated%'` lists
# MAGIC the ones running purely on assumption.
# MAGIC
# MAGIC ## Known-expected states
# MAGIC - `capability_error_rate` CRITICAL: `dsa_optimize` on `test-bot-claude-v1` succeeds only when
# MAGIC   a call finishes under the ~75 s client timeout — 1 in 22 recently. The one success took
# MAGIC   70.7 s. `dsa_compare` on `vertex26` handles a larger prompt in 3.4 s.
# MAGIC - `handover_delivery_rate` / `_new_failure`: 15 handovers never reached
# MAGIC   `PFS3_ISH_SME@lists.lilly.com` — 9.3 % of send attempts since Jun 19, currently 12.5 %
# MAGIC   over 7 days. Acknowledged in `obs_incidents`, so the rate check watches for deterioration.
# MAGIC - `shift_context_missing` WARN permanently until the `dsa_*` capabilities populate
# MAGIC   `shift_date` / `shift_type` / `batch_nbr`. All rows currently write NULL or ''.
# MAGIC - `hallucination_signal_liveness` WARN because `vg_verdict` is NULL on every row — correct,
# MAGIC   since `verify_grounding` does not exist. Layers 2 and 3 carry the signal.
# MAGIC - `latency_anomaly` cannot fire until a capability reaches `n>=30` AND `span>=7d`. Expected
# MAGIC   to become live around 2026-08-25 as `dsa_copilot` accumulates post-blackout history.
# MAGIC - `rapid_human_correction` parked in P1.8: the `dsa_*` capabilities emit no shift/batch
# MAGIC   identity, so there is nothing to correlate. `ish_entity_dim` retained for future use.
# MAGIC
# MAGIC No third-party packages needed; `requests` ships with DBR. If you ever add one, pull it from
# MAGIC the Lilly JFrog Artifactory index, not public PyPI.

# COMMAND ----------

dbutils.widgets.text("env", "dev")
dbutils.widgets.text("lookback_minutes", "60")

ENV = dbutils.widgets.get("env").strip()
LOOKBACK = int(dbutils.widgets.get("lookback_minutes") or 60)
WEBHOOK = dbutils.secrets.get(scope="obs-alerting", key="teams-webhook")
CAT = "mq_gmdf_dev.oil_obs"

if not ENV:
    raise ValueError("Job parameter 'env' is empty. Define it at the job level.")

print(f"env={ENV}  lookback_minutes={LOOKBACK}  webhook_configured={bool(WEBHOOK)}")

# COMMAND ----------

import json
from datetime import datetime, timezone

results = []  # name, severity, value, detail


def check(name, sql, severity_fn, detail_fn=None):
    """Run a scalar-ish check. Missing table -> UNAVAILABLE (a finding, not a crash)."""
    try:
        rows = spark.sql(sql).collect()
    except Exception as e:  # noqa: BLE001 - any failure should surface as a finding
        results.append({"name": name, "severity": "UNAVAILABLE", "value": None,
                        "detail": str(e).split("\n")[0][:300]})
        return
    if not rows:
        results.append({"name": name, "severity": "OK", "value": 0, "detail": "no rows"})
        return
    row = rows[0].asDict()
    detail = detail_fn(row) if detail_fn else json.dumps(
        {k: (str(v) if v is not None else None) for k, v in row.items()})
    results.append({"name": name, "severity": severity_fn(row), "value": row, "detail": detail})


gt0   = lambda r: "CRITICAL" if (r.get("n") or 0) > 0 else "OK"  # noqa: E731
warn0 = lambda r: "WARN"     if (r.get("n") or 0) > 0 else "OK"  # noqa: E731

# COMMAND ----------

# MAGIC %md ## Persist findings to obs_incidents
# MAGIC
# MAGIC Runs before the checks so `unacknowledged_critical` sees this run's findings.
# MAGIC MERGE on `(detector, source_row_id)` means re-detecting increments `detection_count` rather
# MAGIC than duplicating, and acknowledgement survives across runs.
# MAGIC
# MAGIC Any `SKIPPED` line is a column-name mismatch in `INCIDENT_SOURCES` — fix the list rather
# MAGIC than ignoring it, or that detector's findings are never persisted.

# COMMAND ----------

INCIDENT_SOURCES = [
    # detector, source table, id column, capability column, severity, payload columns, extra WHERE
    # latency: only 'anomaly' is CRITICAL. baseline_unreliable and no_baseline are WARN-level
    # conditions reported separately — recording them as CRITICAL overstated severity, including a
    # 70,657 ms dsa_optimize call that had no baseline to judge against.
    ("latency_anomaly",    "latency_anomaly_findings",   "finding_signature",
     "capability",          "CRITICAL",
     ["model_config", "latency_verdict", "worst_latency_ms", "baseline_p95_ms",
      "anomaly_bound_ms", "anomaly_count", "latest_called_at"],
     "WHERE latency_verdict = 'anomaly'"),
        # Keyed on violation_signature, not b.id: keying on the call made every hourly poll from
    # dsa_copilot_step a new incident with notified_at IS NULL, so the 24h suppression in §9.4
    # never had two rows it could recognise as the same thing. 529 cards for one config.
    # Source is transport_violation_signatures because MERGE requires unique source keys.
        # Silenced to WARN (not notified to Teams) 2026-08-31: runtime_allowlist/capability_registry
    # don't yet reflect real production transport/model_config combinations with confidence --
    # test traffic keeps tripping this as CRITICAL, which is noise, not signal, for developers.
    # Re-promote to CRITICAL once the registry-drift fix (see threshold_basis / verification
    # notebook) gives this detector a ground truth worth paging on.
    ("runtime_violation",        "transport_violation_signatures", "violation_signature",
     "capability",               "WARN",
     ["model_config", "transport", "scheduler_run", "violation_type", "violation_tier",
      "calls_in_window", "owner"], "WHERE violation_tier = 'immediate'"),
    ("runtime_violation_digest", "transport_violation_signatures", "violation_signature",
     "capability",               "WARN",
     ["model_config", "transport", "scheduler_run", "violation_type", "violation_tier",
      "calls_in_window", "owner"], "WHERE violation_tier = 'digest'"),
    # WARN persists to obs_incidents but is never notified (§9.4 filters severity='CRITICAL'),
    # which is exactly the digest queue. obs_weekly_runtime_digest reads it.
    ("handover_delivery",  "handover_delivery_failures", "ish_row_id",
     None,                  "CRITICAL",
     ["failure_reason", "shift_date", "batch_nbr"], ""),
    ("hallucination_high", "hallucination_signal",       "row_id",
     "verified_capability", "CRITICAL",
     ["hallucination_risk", "ungrounded_token_count", "resp_vs_prompt_similarity"],
     "WHERE hallucination_risk = 'high'"),
    ("blank_output", "blank_output_findings", "finding_signature",
     "capability", "CRITICAL",
     ["model_config", "blank_rate_window", "blank_count_window", "total_calls_window",
      "latest_hour"], ""),
    ("schema_field_missing", "response_schema_drift", "finding_signature",
     "capability", "CRITICAL",
     ["field_name", "drift_type", "baseline_presence_rate", "current_present", "current_rows"],
     "WHERE drift_type = 'field_missing'"),
    # These two are aggregate-rate detectors, not per-row findings, so they were computed by
    # check() and printed CRITICAL but never persisted here — meaning they could never appear
    # in obs_incidents and could never reach the Teams-notify query below. Each collapses to a
    # single incident (constant source_row_id) that updates in place as the rate changes, same
    # dedup shape as a persistent outage.
    ("capability_error_rate_sustained", "capability_error_rate_alert",
     "concat(capability,'/',model_config)", "capability", "CRITICAL",
     ["model_config", "error_rate_6h", "calls_6h"], ""),
    ("handover_delivery_rate", "handover_delivery_rate", "'handover_delivery_rate_global'",
     None,                     "CRITICAL",
     ["failure_pct_7d", "sent_ok", "failed", "last_attempt"],
     "WHERE failure_pct_7d > 20 AND (sent_ok + failed) >= 10"),
]

# GMDF backtracking for the Teams card's "Where to look" block. Each of this detector's own
# source tables (blank_output_findings, response_schema_drift, latency_anomaly_findings,
# capability_error_rate_alert) is CREATE OR REPLACE'd over a rolling window every run, so a
# query against it goes stale within hours -- confirmed empirically (blank_output_findings had
# 0 rows minutes after producing the 'summary' finding notified above). v_llm_bronze /
# v_ish_bronze are append-only, so a query against them still returns the evidence days later.
# handover_delivery and hallucination_high are exceptions: their own findings tables are built
# from the full history / an unwindowed condition, so they stay valid -- pointed straight at
# the bronze row by id there instead of re-deriving a window filter.
BACKTRACK = {
    # Windows anchor to the incident's own last_detected, not current_timestamp() -- an
    # unacknowledged incident re-notifies every 24h (by design) long after the condition itself
    # stopped recurring, so a "now - INTERVAL" filter would find nothing days later. Anchoring to
    # last_detected reconstructs the window as of when this was actually last true.
    #
    # "lineage" is a static pointer (notebook -> task -> table chain) that stays correct even when
    # the query below doesn't resolve to a row -- confirmed 2026-08-31 that dev-environment source
    # data (mq_gmdf_dev.oil.ptof_primary__ai_llm_audit_log) churns between runs from something
    # outside this pipeline, so a specific historical row is not guaranteed to still be there. The
    # query itself is correct and will resolve reliably against real, non-churning prod data.
    "handover_delivery": {
        "table": "v_ish_bronze",
        "where": lambda cap, p, row_id, last_detected: f"id = '{row_id}'",
        "lineage": "ptof_obs_behavioral_correlation.ipynb -> handover_delivery_failures -> "
                   "v_ish_bronze -> oil.ptof_primary__ish_audit_log",
    },
    "handover_delivery_rate": {
        "table": "v_ish_bronze",
        "lineage": "ptof_obs_behavioral_correlation.ipynb -> handover_delivery_rate -> v_ish_bronze -> oil.ptof_primary__ish_audit_log",
        "where": lambda cap, p, row_id, last_detected: (
            "entity_type = 'HandoverEmail' AND ts BETWEEN "
            f"TIMESTAMP'{last_detected}' - INTERVAL 7 DAYS AND TIMESTAMP'{last_detected}'"),
        "order_by": "ts DESC",
    },
    "capability_error_rate_sustained": {
        "table": "v_llm_bronze",
        "lineage": "ptof_obs_latency_detection.ipynb -> capability_health -> capability_error_rate_alert -> v_llm_bronze -> oil.ptof_primary__ai_llm_audit_log",
        "where": lambda cap, p, row_id, last_detected: (
            f"capability = '{cap}' AND model_config = '{p.get('model_config', '')}' "
            f"AND called_at BETWEEN TIMESTAMP'{last_detected}' - INTERVAL 6 HOURS "
            f"AND TIMESTAMP'{last_detected}' AND success = false"),
        "order_by": "called_at DESC",
    },
    "blank_output": {
        "table": "v_llm_bronze",
        "lineage": "ptof_obs_mal_output.ipynb -> blank_output_incidents -> blank_output_findings -> v_llm_bronze -> oil.ptof_primary__ai_llm_audit_log",
        "where": lambda cap, p, row_id, last_detected: (
            f"capability = '{cap}' AND model_config = '{p.get('model_config', '')}' "
            f"AND is_blank_output = true AND called_at BETWEEN "
            f"TIMESTAMP'{last_detected}' - INTERVAL 6 HOURS AND TIMESTAMP'{last_detected}'"),
        "order_by": "called_at DESC",
    },
    "schema_field_missing": {
        "table": "v_llm_bronze",
        "lineage": "ptof_obs_mal_output.ipynb -> response_schema_drift -> v_llm_bronze -> oil.ptof_primary__ai_llm_audit_log",
        "where": lambda cap, p, row_id, last_detected: (
            f"capability = '{cap}' AND called_at BETWEEN "
            f"TIMESTAMP'{last_detected}' - INTERVAL 24 HOURS AND TIMESTAMP'{last_detected}'"),
        "order_by": "called_at DESC",
        "note": "Field-missing can't be expressed as a plain filter -- inspect response_parsed "
                "on these rows directly.",
    },
    "latency_anomaly": {
        "table": "v_llm_bronze",
        "lineage": "ptof_obs_latency_detection.ipynb -> latency_anomalies -> latency_anomaly_findings -> v_llm_bronze -> oil.ptof_primary__ai_llm_audit_log",
        "where": lambda cap, p, row_id, last_detected: (
            f"capability = '{cap}' AND model_config = '{p.get('model_config', '')}' "
            f"AND called_at BETWEEN TIMESTAMP'{last_detected}' - INTERVAL 60 MINUTES "
            f"AND TIMESTAMP'{last_detected}'"),
        "order_by": "latency_ms DESC",
    },
    "hallucination_high": {
        "table": "v_llm_bronze",
        "lineage": "ptof_obs_hallucination_detection.ipynb -> hallucination_signal -> v_llm_bronze -> oil.ptof_primary__ai_llm_audit_log",
        "where": lambda cap, p, row_id, last_detected: f"id = '{row_id}'",
    },
}

for detector, table, id_col, cap_col, severity, payload_cols, extra in INCIDENT_SOURCES:
    try:
        cap_expr = cap_col if cap_col else "CAST(NULL AS STRING)"
        payload = ", ".join(f"'{c}', CAST({c} AS STRING)" for c in payload_cols)
        # If a detector's payload wants "owner", join capability_registry once here
        # rather than baking the join into every source table. Wrapped as a derived
        # table so {extra}'s WHERE clause (e.g. "violation_tier = 'immediate'")
        # still resolves unqualified column names against it.
        source_expr = f"{CAT}.{table}"
        if "owner" in payload_cols:
            source_expr = f"""(
                SELECT tbl.*, coalesce(r.owner, 'unassigned') AS owner
                FROM {CAT}.{table} tbl
                LEFT JOIN {CAT}.capability_registry r ON r.capability = tbl.capability
            ) v"""
        spark.sql(f"""
            MERGE INTO {CAT}.obs_incidents t
            USING (
              SELECT '{detector}'             AS detector,
                     CAST({id_col} AS STRING) AS source_row_id,
                     {cap_expr}               AS capability,
                     '{severity}'             AS severity,
                     to_json(map({payload}))  AS signal_payload
              FROM {source_expr} {extra}

            ) s
            ON t.detector = s.detector AND t.source_row_id = s.source_row_id
            WHEN MATCHED THEN UPDATE SET
                t.last_detected   = current_timestamp(),
                t.detection_count = t.detection_count + 1,
                t.signal_payload  = s.signal_payload
            WHEN NOT MATCHED THEN INSERT
                (detector, source_row_id, capability, severity,
                 first_detected, last_detected, detection_count, signal_payload)
              VALUES
                (s.detector, s.source_row_id, s.capability, s.severity,
                 current_timestamp(), current_timestamp(), 1, s.signal_payload)
        """)
        print(f"  emitted: {detector}")
    except Exception as e:  # noqa: BLE001
        print(f"  SKIPPED {detector}: {str(e).split(chr(10))[0][:160]}")

print("Incident emission complete.")

# COMMAND ----------

# MAGIC %md ## Latency

# COMMAND ----------

# Reliable baselines only. is_reliable = (n_samples >= 30 AND baseline_span_days >= 7).
# Nothing qualifies yet — this becomes live as dsa_copilot accumulates 7 days of history.
check(
    "latency_anomaly",
    f"""
    SELECT count(*) AS n, max(la.latency_ms) AS worst_ms,
           concat_ws(',', collect_set(la.capability)) AS capabilities
    FROM {CAT}.latency_anomalies la
    JOIN {CAT}.capability_latency_baseline base USING (capability)
    WHERE la.latency_verdict = 'anomaly'
      AND base.is_reliable = true
      AND la.called_at >= current_timestamp() - INTERVAL {LOOKBACK} MINUTES
    """,
    gt0,
)

# Same anomaly definition as above, but against a baseline that hasn't reached
# n>=30 / span>=7d yet -- WARN only, since this capability's history is still too thin
# to trust the anomaly bound.
check(
    "latency_anomaly_unreliable_baseline",
    f"""
    SELECT count(*) AS n,
           concat_ws(',', collect_set(concat(la.capability,'(n=',base.n_samples,
                                            ',span=',base.baseline_span_days,'d)'))) AS capabilities
    FROM {CAT}.latency_anomalies la
    JOIN {CAT}.capability_latency_baseline base USING (capability)
    WHERE la.latency_verdict IN ('anomaly','anomaly_fixed_ceiling','baseline_unreliable')
      AND base.is_reliable = false
      AND la.called_at >= current_timestamp() - INTERVAL {LOOKBACK} MINUTES
    """,
    warn0,
)

# Successful calls exist with no baseline row at all -- distinguishes 'no history yet'
# from 'anomalous', so a brand-new capability doesn't misread as broken.
check(
    "latency_baseline_missing",
    f"""
    SELECT count(DISTINCT b.capability) AS n,
           concat_ws(',', collect_set(b.capability)) AS capabilities
    FROM {CAT}.v_llm_bronze b
    LEFT JOIN {CAT}.capability_latency_baseline base USING (capability)
    WHERE b.called_at >= current_timestamp() - INTERVAL {LOOKBACK} MINUTES
      AND b.success = true
      AND b.is_credential_fastfail = false
      AND base.capability IS NULL
    """,
    warn0,
)

# A call over a fixed ceiling deserves visibility even with no baseline. dsa_optimize's one
# success took 70,657 ms and classified as 'no_baseline', so nothing flagged it.
check(
    "latency_fixed_ceiling",
    f"""
    SELECT count(*) AS n, max(latency_ms) AS worst_ms,
           concat_ws(',', collect_set(capability)) AS capabilities
    FROM {CAT}.v_llm_bronze
    WHERE success = true
      AND latency_ms > 30000
      AND called_at >= current_timestamp() - INTERVAL {LOOKBACK} MINUTES
    """,
    warn0,
)

# COMMAND ----------

# MAGIC %md ## Capability health

# COMMAND ----------

# Deliberately does NOT filter success = true. Every other detector does, which is why a
# 153-call outage on dsa_optimize was invisible for four days.
check(
    "capability_error_rate",
    f"""
    SELECT count(*) AS n,
           concat_ws(', ', collect_set(concat(capability,'/',model_config,'=',
                      cast(round(error_rate,2) AS STRING)))) AS offenders
    FROM {CAT}.capability_health
    WHERE hour >= date_trunc('HOUR', current_timestamp() - INTERVAL {LOOKBACK} MINUTES)
      AND ((error_rate = 1.0  AND total_calls >= 5)
        OR (error_rate > 0.20 AND total_calls >= 10))
    """,
    gt0,
)

# Sustained failure over 6h. The per-hour check applies its volume floor to a single hour, which
# let a 100%-failing capability read OK whenever the most recent hour happened to be quiet.
check(
    "capability_error_rate_sustained",
    f"""
    SELECT count(*) AS n,
           concat_ws(', ', collect_set(concat(capability,'/',model_config,'=',
                      cast(error_rate_6h AS STRING),' over ',
                      cast(calls_6h AS STRING),' calls'))) AS offenders
    FROM {CAT}.capability_error_rate_alert
    """,
    gt0,
)
# Time since last call, not daily count. dsa_copilot grace raised 18 -> 26 h after two false
# breaches (19.5 h, 22.7 h) and zero true ones: it is human-driven with a ~10 h overnight gap.
check(
    "capability_silence",
    f"""
    SELECT count(*) AS n,
           concat_ws(', ', collect_set(concat(capability,'=',
                      cast(hours_since_last_call AS STRING),'h'))) AS quiet
    FROM {CAT}.capability_silence
    """,
    warn0,
)

# Zero v_llm_bronze rows in the last 2h means ingestion itself has stopped -- catches a
# total pipeline outage upstream of every other detector, which all assume rows exist
# to filter.
check(
    "pipeline_heartbeat",
    f"""
    SELECT CASE WHEN count(*) = 0 THEN 1 ELSE 0 END AS n, count(*) AS rows_last_2h
    FROM {CAT}.v_llm_bronze
    WHERE called_at >= current_timestamp() - INTERVAL 2 HOURS
    """,
    gt0,
)

# dsa_* capabilities writing NULL/'' shift identity -- Phase 4 item 10 of
# agent_obs_implementation_plan.md (registry/identity drift, still open). Standing
# WARN, not notified, until that gap closes.
check(
    "shift_context_missing",
    f"""
    SELECT count(*) AS n,
           concat_ws(', ', collect_set(concat(capability,'=',
                      cast(total_calls AS STRING),' calls'))) AS affected
    FROM {CAT}.shift_context_missing
    """,
    warn0,
)

# COMMAND ----------

# MAGIC %md ## Failures, write lag, blank output

# COMMAND ----------

# Fastfail short-circuits (credential/auth rejected before a real call was attempted).
# capability_error_rate can miss this shape if the fastfail path never lands on a plain
# success=false row.
check(
    "credential_outage",
    f"""
    SELECT coalesce(sum(fastfail_calls), 0) AS n,
           concat_ws(',', collect_set(model_config)) AS pools
    FROM {CAT}.credential_fastfail_daily
    WHERE day >= date_trunc('DAY', current_timestamp())
    """,
    gt0,
)

# Reads p95_ingest_only_s, which subtracts latency_ms. write_lag_s spans the whole call, so a
# 70 s generation registered as 73 s of "lag" while true ingestion was 2.3 s.
# Threshold tightened 30 -> 10 s: observed range is 2.1-2.3 s, so 30 needed a 13x degradation.
check(
    "write_lag",
    f"""
    SELECT p95_ingest_only_s, ingest_over_60s_rows, p95_s, total_rows, hour,
           CASE WHEN (p95_ingest_only_s > 10 OR ingest_over_60s_rows > 0)
                 AND total_rows >= 10
                THEN 1 ELSE 0 END AS n
    FROM {CAT}.write_lag_daily
    WHERE hour < date_trunc('HOUR', current_timestamp())
    ORDER BY hour DESC LIMIT 1
    """,
    gt0,
)

# Response content empty on calls that formally succeeded -- invisible to every
# error-rate-based detector above, since a blank response never raises or fails.
check(
    "blank_output",
    f"""
    SELECT count(*) AS n, max(blank_output_rate) AS worst_rate
    FROM {CAT}.blank_output_incidents
    WHERE hour >= date_trunc('HOUR', current_timestamp() - INTERVAL {LOOKBACK} MINUTES)
      AND blank_output_rate > 0.02
      AND blank_output_count > 3
      AND total_successful_calls >= 10
    """,
    gt0,
)

# COMMAND ----------

# MAGIC %md ## Schema and runtime configuration

# COMMAND ----------

# Any change to a capability's response shape vs. its stored baseline, independent of
# whether a specific field went missing (see schema_field_missing below).
check(
    "schema_drift",
    f"""
    SELECT count(*) AS n, concat_ws(',', collect_set(capability)) AS capabilities
    FROM {CAT}.response_schema_drift
    WHERE schema_changed = true
    """,
    warn0,
)

# Coverage gap, not a finding about the capability itself: capabilities with enough
# eligible history (>=20 successful, non-blank calls in 30d) that still have no baseline
# row -- a monitoring blind spot.
check(
    "schema_baseline_coverage",
    f"""
    WITH eligible AS (
      SELECT b.capability
      FROM {CAT}.v_llm_bronze b
      JOIN {CAT}.capability_registry r
        ON r.capability = b.capability AND r.active = true
      WHERE b.success = true AND b.is_blank_output = false
        AND b.called_at >= current_timestamp() - INTERVAL 30 DAYS
      GROUP BY b.capability HAVING count(*) >= 20
    )
    SELECT (SELECT count(*) FROM {CAT}.response_schema_baseline) AS baselined,
           (SELECT count(*) FROM eligible)                       AS eligible,
           CASE WHEN (SELECT count(*) FROM {CAT}.response_schema_baseline)
                   < (SELECT count(*) FROM eligible)
                THEN 1 ELSE 0 END AS n
    """,
    warn0,
)

# A field that was reliably present has disappeared. Downstream code reading it as
# null/false instead of unknown is the risk this exists to catch.
check(
    "schema_field_missing",
    f"""
    SELECT count(*) AS n,
           concat_ws(', ', collect_set(concat(capability,': ',field_name))) AS detail
    FROM {CAT}.response_schema_drift
    WHERE drift_type = 'field_missing'
    """,
    gt0,
)

# An empty allowlist makes every call read as unknown_capability. Distinguish "the reference table
# is broken" from "traffic is violating it".
check(
    "runtime_allowlist_populated",
    f"""
    SELECT CASE WHEN count(*) = 0 THEN 1 ELSE 0 END AS n,
           count(*) AS rows_for_env
    FROM {CAT}.runtime_allowlist WHERE environment = '{ENV}'
    """,
    gt0,
)

# Reads the detector table rather than re-deriving the join, so there is one definition of a
# violation. n counts CONFIGURATIONS, not calls: n=21 previously meant 21 polls of one config.
check(
    "runtime_violation",
    f"""
    SELECT count(*) AS n,
           max(calls_in_window) AS worst_call_count,
           concat_ws(', ', collect_set(concat(capability,'/',model_config,'/',
                                              transport,'/',scheduler_run))) AS offenders
    FROM {CAT}.transport_violation_signatures
    WHERE violation_tier = 'immediate'
    """,
    warn0,  # silenced from gt0/CRITICAL -- see INCIDENT_SOURCES comment above
)

# The digest queue, visible in task output but never notified. If this climbs and stays up,
# the weekly digest is not being dispositioned.
check(
    "runtime_violation_digest_backlog",
    f"""
    SELECT count(*) AS n,
           concat_ws(', ', collect_set(concat(capability,'/',
                      get_json_object(signal_payload, '$.model_config')))) AS pending
    FROM {CAT}.obs_incidents
    WHERE detector = 'runtime_violation_digest'
      AND acknowledged_at IS NULL AND resolved_at IS NULL
    """,
    warn0,
)

# COMMAND ----------

# MAGIC %md ## Incident state

# COMMAND ----------

# Backstop rollup: any CRITICAL incident still open and unacknowledged, regardless of
# which detector raised it -- catches anything already flagged above that nobody has
# acted on.
check(
    "unacknowledged_critical",
    f"""
    SELECT count(*) AS n,
           concat_ws(', ', collect_set(concat(detector,':',coalesce(capability,'-')))) AS detail,
           min(first_detected) AS oldest
    FROM {CAT}.obs_incidents
    WHERE severity = 'CRITICAL'
      AND acknowledged_at IS NULL
      AND resolved_at IS NULL
    """,
    gt0,
)

# Age-based, not detection_count. detection_count counts RUNS that saw the finding, not distinct
# occurrences — testing showed 24x in four hours, so a count threshold of 20 would trip within
# two hours of 5-minute triggering and mean nothing. Three days unacknowledged is a real signal.
check(
    "long_running_incident",
    f"""
    SELECT count(*) AS n,
           concat_ws(', ', collect_set(concat(detector,' open ',
                      cast(datediff(current_timestamp(), first_detected) AS STRING),'d'))) AS detail
    FROM {CAT}.obs_incidents
    WHERE resolved_at IS NULL
      AND acknowledged_at IS NULL
      AND first_detected <= current_timestamp() - INTERVAL 3 DAYS
    """,
    warn0,
)

# COMMAND ----------

# MAGIC %md ## Hallucination and handover delivery

# COMMAND ----------

# The primary hallucination signal that reaches Teams. See
# hallucination_signal_liveness below for whether this signal is even being computed.
check(
    "hallucination_high",
    f"""
    SELECT count(*) AS n,
           concat_ws(',', collect_set(verified_capability)) AS capabilities
    FROM {CAT}.hallucination_signal
    WHERE hallucination_risk = 'high'
      AND called_at >= current_timestamp() - INTERVAL {LOOKBACK} MINUTES
    """,
    gt0,
)

# total = 0 is itself the finding. with_verdict = 0 is EXPECTED — verify_grounding does not exist.
check(
    "hallucination_signal_liveness",
    f"""
    SELECT count(*) AS total,
           count_if(vg_verdict IS NOT NULL) AS with_verdict,
           CASE WHEN count(*) = 0
                  OR (count(*) > 0 AND count_if(vg_verdict IS NOT NULL) = 0)
                THEN 1 ELSE 0 END AS n
    FROM {CAT}.hallucination_signal
    """,
    warn0,
)

# Share of recent calls the grounding layers couldn't classify at all. A high unverified
# rate means the signal is going blind, not that hallucination is absent.
check(
    "hallucination_unverified_rate",
    f"""
    SELECT count(*) AS total,
           count_if(hallucination_risk = 'unverified') AS unverified,
           CASE WHEN count(*) >= 10
                 AND count_if(hallucination_risk = 'unverified') * 1.0 / count(*) > 0.5
                THEN 1 ELSE 0 END AS n
    FROM {CAT}.hallucination_signal
    WHERE called_at >= current_timestamp() - INTERVAL 24 HOURS
    """,
    warn0,
)

# Rate, not count. The 15 historical failures are acknowledged; this fires only if delivery gets
# worse than the established 9.3 % baseline. Floor of 10 attempts because a 7-day window at
# ~2 attempts/day swings wildly on small numbers (daily rates hit 33 % and 100 % on one failure).
check(
    "handover_delivery_rate",
    f"""
    SELECT failure_pct_7d, sent_ok, failed, last_attempt,
           CASE WHEN failure_pct_7d > 20 AND (sent_ok + failed) >= 10
                THEN 1 ELSE 0 END AS n
    FROM {CAT}.handover_delivery_rate
    """,
    gt0,
)

# Any failure not yet acknowledged. Unlike the rate, this catches a single recurrence — which
# matters because each one is a shift handover nobody received.
check(
    "handover_delivery_new_failure",
    f"""
    SELECT count(*) AS n, max(attempted_at) AS latest
    FROM {CAT}.handover_delivery_failures f
    WHERE NOT exists (SELECT 1 FROM {CAT}.obs_incidents i
                      WHERE i.detector = 'handover_delivery'
                        AND i.source_row_id = f.ish_row_id
                        AND i.acknowledged_at IS NOT NULL)
    """,
    gt0,
)

# COMMAND ----------

order = {"CRITICAL": 0, "UNAVAILABLE": 1, "WARN": 2, "OK": 3}
results.sort(key=lambda r: order.get(r["severity"], 9))

for r in results:
    print(f"{r['severity']:<12} {r['name']:<40} {r['detail']}")

criticals   = [r for r in results if r["severity"] == "CRITICAL"]
unavailable = [r for r in results if r["severity"] == "UNAVAILABLE"]
warns       = [r for r in results if r["severity"] == "WARN"]

# COMMAND ----------

# MAGIC %md ## Teams card
# MAGIC
# MAGIC A card naming a table and dumping JSON tells a responder nothing. Each detector carries a
# MAGIC plain-language label, what the condition means, and where to start looking.
# MAGIC
# MAGIC The triage hints encode what this build established — the ~75 s client ceiling, the 9.3 %
# MAGIC handover baseline, the 0.69-0.89 similarity band, that projection capabilities legitimately
# MAGIC emit novel numbers. Useful now; will go stale. Revisit when the underlying facts change.

# COMMAND ----------

DETECTOR_META = {
    "handover_delivery": {
        "label": "Shift handover email failed to send",
        "what": "An automated shift handover never reached PFS3_ISH_SME@lists.lilly.com. "
                "The audit row exists; the email does not.",
        "triage": [
            "getaddrinfo failure = DNS resolution to the mail host, not an app bug",
            "Check whether the affected shift was handed over by other means",
            "Baseline is ~9.3% of send attempts since Jun 19 — check handover_delivery_rate to "
            "see whether this is drift or the standing rate",
        ],
    },
    "capability_error_rate": {
        "label": "Capability failing at or near 100%",
        "what": "An LLM capability is failing most or all of its calls. Every other detector "
                "filters success = true, so this is the only one that sees a total outage.",
        "triage": [
            "Compare model_config against sibling capabilities — a mismatch is the usual cause",
            "Check error_class in v_llm_bronze: timeout, auth, upstream_5xx or connection",
            "Timeouts near a round number (~75s) suggest a client ceiling, not a slow model",
        ],
    },
    "latency_anomaly": {
        "label": "Call latency beyond the capability's baseline",
        "what": "A successful call exceeded p95 + 3xIQR for its capability, measured against a "
                "baseline with at least 30 samples over at least 7 days.",
        "triage": [
            "Check user_prompt_chars — payload growth is the usual leading indicator",
            "Compare against capability_latency_baseline to see how far outside it sits",
            "latency_failures shows whether timeouts are rising alongside it",
        ],
    },
        "runtime_violation": {
        "label": "Capability ran on an unapproved transport or model config",
        "what": "A call used a transport or model_config that is sanctioned for no capability "
                "at all in this environment. New capabilities on already-sanctioned "
                "infrastructure go to the weekly digest instead of here.",
        "triage": [
            "Confirm runtime_allowlist is populated for this env before assuming the traffic "
            "is wrong — an empty allowlist is tiered to digest, but a partial one is not",
            "A test model config reaching a live scheduler run is the usual cause",
            "calls_in_window tells you whether this is one stray call or a running poll",
        ],
    },
    "hallucination_high": {
        "label": "Response may contain ungrounded content",
        "what": "A response scored high risk across the grounding layers: unsupported numeric "
                "tokens, low similarity to its prompt, or a failed verification verdict.",
        "triage": [
            "Read ungrounded_tokens — computed values are expected for projection capabilities",
            "Only summarization capabilities are is_gxp_relevant; chat and scoring legitimately "
            "emit novel numbers",
            "Compare resp_vs_prompt_similarity against the 0.69-0.89 observed band",
        ],
    },
    "blank_output": {
        "label": "Agent returned a blank response",
        "what": "A capability's calls formally succeeded (no error, no exception) but the "
                "response content was empty often enough to exceed the blank-rate floor. "
                "Invisible to every error-rate-based detector in this system.",
        "triage": [
            "Check whether this is a prompt/template regression or an upstream input problem",
            "blank_rate_window / blank_count_window / total_calls_window show how bad and how big",
            "A blank handover or summary reaching a real person is worse than an error — nobody "
            "else is watching for this",
        ],
    },
    "schema_field_missing": {
        "label": "Expected output field went missing",
        "what": "A field that was reliably present in this capability's response schema has "
                "disappeared. Downstream automation reading a missing field as null/false "
                "instead of unknown is the real risk here.",
        "triage": [
            "Check for a recent prompt-template edit or model swap on this capability",
            "baseline_presence_rate tells you how reliably the field used to appear",
            "Known gap: this detector can't see dsa_copilot_step / dsa_session_summary / "
            "probe / saa_insight yet -- they aren't in capability_registry",
        ],
    },
        "capability_error_rate_sustained": {
        "label": "Capability failing continuously over hours",
        "what": "Aggregated over 6 hours rather than per hour, so a sustained outage cannot hide "
                "behind a single quiet hour.",
        "triage": [
            "Compare model_config against sibling capabilities on the same surface",
            "hours_fully_failed in capability_error_rate_alert — all hours failed means config, "
            "not load",
            "Check error_class: timeout, auth, upstream_5xx or connection",
        ],
    },
    "handover_delivery_rate": {
        "label": "Shift handover delivery rate deteriorating",
        "what": "Handover email failures over the trailing 7 days have exceeded the established "
                "baseline, with enough attempts (>=10) for the rate to be meaningful.",
        "triage": [
            "Compare against the ~9.3% historical baseline noted under handover_delivery — "
            "this fires only once it gets worse than that",
            "Check handover_delivery_failures for the underlying getaddrinfo/DNS pattern",
            "sent_ok / failed counts are in the payload below",
        ],
    },
}


def _age(first_detected):
    """Human-readable age. Replaces 'seen Nx', which counted alert runs rather than occurrences
    and overstated severity — 24x in four hours was just the alert running often."""
    delta = datetime.now(timezone.utc) - first_detected.replace(tzinfo=timezone.utc)
    d, h = delta.days, delta.seconds // 3600
    if d:
        return f"{d}d {h}h old"
    m = (delta.seconds % 3600) // 60
    return f"{h}h {m}m old" if h else f"{m}m old"


def _job_run_url():
    """Link back to this job run. Absent in interactive runs, which is fine."""
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        host = ctx.tags().get("browserHostName").get()
        job_id = ctx.tags().get("jobId").get()
        run_id = ctx.tags().get("multitaskParentRunId").get()
        return f"https://{host}/jobs/{job_id}/runs/{run_id}"
    except Exception:  # noqa: BLE001
        return None


def _facts_from_payload(payload):
    """signal_payload as labelled facts rather than a raw JSON blob."""
    try:
        d = json.loads(payload) if payload else {}
    except Exception:  # noqa: BLE001
        return [{"title": "payload", "value": str(payload)[:200]}]
    pretty = {
        "failure_reason": "Reason", "shift_date": "Shift date", "batch_nbr": "Batch",
        "latency_ms": "Latency (ms)", "latency_verdict": "Verdict", "p95_ms": "Baseline p95 (ms)",
        "model_config": "Model config", "transport": "Transport",
        "scheduler_run": "Scheduler run", "violation_type": "Violation",
        "hallucination_risk": "Risk", "ungrounded_token_count": "Ungrounded tokens",
        "resp_vs_prompt_similarity": "Similarity", "owner": "Owner",
        "error_rate_6h": "Error rate (6h)", "calls_6h": "Calls (6h)",
        "failure_pct_7d": "Failure % (7d)", "sent_ok": "Sent OK", "failed": "Failed",
        "last_attempt": "Last attempt",
        "blank_rate_window": "Blank rate", "blank_count_window": "Blank count",
        "total_calls_window": "Total calls (window)", "latest_hour": "Latest hour",
        "field_name": "Field", "drift_type": "Drift type",
        "baseline_presence_rate": "Baseline presence rate", "current_present": "Currently present",
        "current_rows": "Current rows",
        "worst_latency_ms": "Worst latency (ms)", "baseline_p95_ms": "Baseline p95 (ms)",
        "anomaly_bound_ms": "Anomaly bound (ms)", "anomaly_count": "Anomaly count",
        "latest_called_at": "Latest call",
    }
    return [{"title": pretty.get(k, k), "value": str(v)[:180]}
            for k, v in d.items() if v not in (None, "", "null")]

# COMMAND ----------

def post_teams(incidents, broken):
    """POST a triage-oriented Adaptive Card to the Teams webhook (Workflows-based, expect 202).

    try/except on purpose: a Teams outage must not fail the alert task, or a notification
    problem becomes indistinguishable from a monitoring problem.
    """
    if not WEBHOOK:
        print("No webhook configured; skipping notification.")
        return
    import requests

    severity = "DETECTOR BROKEN" if broken else "CRITICAL"
    headline = (f"{len(broken)} detector(s) stopped working" if broken
                else f"{len(incidents)} critical finding(s)")

    body = [{
        "type": "Container", "style": "attention", "bleed": True,
        "items": [
            {"type": "TextBlock", "size": "Large", "weight": "Bolder",
             "text": f"{severity} — {headline}"},
            {"type": "TextBlock", "spacing": "None", "isSubtle": True, "wrap": True,
             "text": f"ISH agent observability · env **{ENV}** · "
                     f"{datetime.now(timezone.utc):%Y-%m-%d %H:%M} UTC"},
        ],
    }]

    # Broken detectors first: monitoring has stopped, which outranks anything it found.
    for r in broken:
        body.append({
            "type": "Container", "style": "attention", "separator": True,
            "items": [
                {"type": "TextBlock", "weight": "Bolder", "wrap": True,
                 "text": f"Detector stopped: {r['name']}"},
                {"type": "TextBlock", "wrap": True, "isSubtle": True,
                 "text": "This check could not run — a table is missing or a query threw. "
                         "Nothing is watching this condition until it is fixed."},
                {"type": "TextBlock", "wrap": True, "fontType": "Monospace",
                 "text": r["detail"][:400]},
            ],
        })

    # Group by detector so fifteen failures of one kind read as one problem.
    by_detector = {}
    for r in incidents:
        by_detector.setdefault(r["detector"], []).append(r)

    for detector, rows in by_detector.items():
        meta = DETECTOR_META.get(detector, {})
        oldest = min(r["first_detected"] for r in rows)
        caps = sorted({x["capability"] for x in rows if x["capability"]})

        items = [
            {"type": "TextBlock", "size": "Medium", "weight": "Bolder", "wrap": True,
             "text": meta.get("label", detector)},
            {"type": "TextBlock", "spacing": "None", "isSubtle": True, "wrap": True,
             "text": f"`{detector}`"
                     + (f" · {', '.join(caps)}" if caps else "")
                     + f" · {len(rows)} occurrence(s) · oldest {_age(oldest)}"},
        ]
        if meta.get("what"):
            items.append({"type": "TextBlock", "wrap": True, "text": meta["what"]})

        if detector == "runtime_violation":
            # One new endpoint appearing under N capabilities is one operational event,
            # not N separate things to read. Group by the resource (transport +
            # model_config) rather than by capability, same principle as the weekly
            # digest's per-capability grouping, one level up.
            by_resource = {}
            for r in rows:
                p = json.loads(r["signal_payload"]) if r["signal_payload"] else {}
                key = (p.get("transport", "?"), p.get("model_config", "?"))
                by_resource.setdefault(key, []).append((r, p))

            for (transport, model_config), entries in sorted(by_resource.items()):
                resource_caps = sorted({r["capability"] for r, _ in entries if r["capability"]})
                total_calls = sum(int(p.get("calls_in_window", 0) or 0) for _, p in entries)
                resource_oldest = min(r["first_detected"] for r, _ in entries)
                owners = sorted({p.get("owner", "unassigned") for _, p in entries
                                 if p.get("owner") not in (None, "", "null")})

                items.append({"type": "TextBlock", "weight": "Bolder", "size": "Small",
                              "spacing": "Medium",
                              "text": f"New resource: {model_config} ({transport})"})
                items.append({"type": "FactSet", "facts": [
                    {"title": "Capabilities using it", "value": ", ".join(resource_caps) or "-"},
                    {"title": "Owner(s)",               "value": ", ".join(owners) or "unassigned"},
                    {"title": "Total calls",            "value": str(total_calls)},
                    {"title": "First seen",             "value": _age(resource_oldest)},
                ]})
        else:
            newest = max(rows, key=lambda x: x["first_detected"])
            facts = _facts_from_payload(newest["signal_payload"])
            if facts:
                items.append({"type": "TextBlock", "weight": "Bolder", "size": "Small",
                              "spacing": "Medium",
                              "text": "Most recent occurrence" if len(rows) > 1 else "Details"})
                items.append({"type": "FactSet", "facts": facts})

            # GMDF backtracking: a copy-paste query against the durable oil-layer bronze table,
            # not this detector's own (rolling-window, CREATE OR REPLACE'd) findings table -- see
            # the BACKTRACK comment above for why that table would likely already be empty.
            bt = BACKTRACK.get(detector)
            if bt:
                payload_dict = json.loads(newest["signal_payload"]) if newest["signal_payload"] else {}
                where_clause = bt["where"](newest["capability"], payload_dict,
                                           newest["source_row_id"], newest["last_detected"])
                query = f"SELECT * FROM {CAT}.{bt['table']} WHERE {where_clause}"
                if bt.get("order_by"):
                    query += f" ORDER BY {bt['order_by']}"
                query += " LIMIT 20;"
                items.append({"type": "TextBlock", "weight": "Bolder", "size": "Small",
                              "spacing": "Medium", "text": "Where to look"})
                if bt.get("lineage"):
                    items.append({"type": "TextBlock", "wrap": True, "spacing": "None",
                                  "isSubtle": True, "text": bt["lineage"]})
                if bt.get("note"):
                    items.append({"type": "TextBlock", "wrap": True, "spacing": "None",
                                  "isSubtle": True, "text": bt["note"]})
                items.append({"type": "TextBlock", "wrap": True, "spacing": "None",
                              "fontType": "Monospace", "text": query})

        if meta.get("triage"):
            items.append({"type": "TextBlock", "weight": "Bolder", "size": "Small",
                          "spacing": "Medium", "text": "Where to start"})
            items += [{"type": "TextBlock", "wrap": True, "spacing": "None", "isSubtle": True,
                       "text": f"\u2022 {t}"} for t in meta["triage"]]

        # Real detector name substituted, so this is copy-and-run rather than copy-and-edit.
        items.append({
            "type": "TextBlock", "wrap": True, "spacing": "Medium", "size": "Small",
            "isSubtle": True, "fontType": "Monospace",
            "text": "Acknowledge (suppresses the alert, keeps the record):<br>"
                    f"UPDATE mq_gmdf_dev.oil_obs.obs_incidents SET "
                    f"acknowledged_by='you@lilly.com', acknowledged_at=current_timestamp() "
                    f"WHERE detector='{detector}' AND acknowledged_at IS NULL;",
        })

        body.append({"type": "Container", "separator": True, "spacing": "Medium",
                     "items": items})

    card = {"type": "AdaptiveCard", "version": "1.4", "body": body,
            "$schema": "http://adaptivecards.io/schemas/adaptive-card.json"}

    run_url = _job_run_url()
    if run_url:
        card["actions"] = [{"type": "Action.OpenUrl", "title": "Open job run", "url": run_url}]

    try:
        resp = requests.post(
            WEBHOOK,
            data=json.dumps({"type": "message", "attachments": [{
                "contentType": "application/vnd.microsoft.card.adaptive",
                "content": card}]}),
            headers={"Content-Type": "application/json"}, timeout=30)
        print(f"Teams webhook status={resp.status_code}")
        if resp.status_code >= 400:
            print(f"  response: {resp.text[:300]}")
    except Exception as e:  # noqa: BLE001
        print(f"Teams webhook FAILED: {str(e)[:200]}")

# COMMAND ----------

# Notify once per incident, not once per run.
#
# Without this, a persistent finding posts every 5 minutes forever — the alert-fatigue failure this
# design exists to prevent. obs_incidents.notified_at carries the state: an incident is reported
# when first seen, then not again for 24 h unless it is still unacknowledged.
#
# WARN findings are deliberately NOT notified. shift_context_missing and
# hallucination_signal_liveness are standing conditions; posting them would drown the channel.
# They stay visible in the task output above and in obs_incidents.
try:
    to_notify = spark.sql(f"""
        SELECT detector, source_row_id, capability, severity, first_detected, last_detected,
               detection_count, signal_payload
        FROM {CAT}.obs_incidents
        WHERE severity = 'CRITICAL'
          AND acknowledged_at IS NULL
          AND resolved_at IS NULL
          AND (notified_at IS NULL
               OR notified_at < current_timestamp() - INTERVAL 24 HOURS)
        ORDER BY first_detected
    """).collect()
except Exception as e:  # noqa: BLE001
    print(f"incident notify query failed: {str(e).split(chr(10))[0][:160]}")
    to_notify = []

if to_notify or unavailable:
    post_teams(to_notify, unavailable)
    if to_notify:
        spark.sql(f"""
            UPDATE {CAT}.obs_incidents
            SET notified_at = current_timestamp()
            WHERE severity = 'CRITICAL'
              AND acknowledged_at IS NULL AND resolved_at IS NULL
              AND (notified_at IS NULL
                   OR notified_at < current_timestamp() - INTERVAL 24 HOURS)
        """)
else:
    print("No new or stale incidents to notify.")

# COMMAND ----------

# Raise ONLY on UNAVAILABLE. Task status means "is monitoring working", not "did it find
# something".
#
# UNAVAILABLE means a table is missing or a query threw: a detector silently stopped monitoring,
# which is the condition that hid every problem in this build. That is worth failing a task over.
# CRITICAL findings are tracked in obs_incidents with acknowledgement state and routed to Teams.
if unavailable:
    raise Exception("ISH observability — DETECTOR BROKEN: " + "; ".join(
        f"{r['name']}={r['severity']}" for r in unavailable))

if criticals:
    print(f"\n{len(criticals)} CRITICAL finding(s) above. Task not failed: findings route via "
          f"obs_incidents and Teams, not task status.")
else:
    print("\nNo CRITICAL findings.")

env=dev  lookback_minutes=60  webhook_configured=True
  emitted: latency_anomaly
  emitted: runtime_violation
  emitted: runtime_violation_digest
  emitted: handover_delivery
  emitted: hallucination_high
Incident emission complete.
CRITICAL     pipeline_heartbeat                       {"n": "1", "rows_last_2h": "0"}
CRITICAL     unacknowledged_critical                  {"n": "11", "detail": "runtime_violation:dsa_copilot_step, runtime_violation:dsa_optimize, runtime_violation:sev2_insight, runtime_violation:summary, runtime_violation:saa_insight", "oldest": "2026-08-27 16:58:00.131376"}
WARN         capability_silence                       {"n": "2", "quiet": "dsa_copilot=30.5h, dsa_optimize=13.2h"}
WARN         shift_context_missing                    {"n": "4", "affected": "dsa_batch_summary=7 calls, dsa_copilot=21 calls, dsa_compare=18 calls, dsa_optimize=383 calls"}
WARN         runtime_violation_digest_backlog         {"n": "4", "pending": "dsa_session_summary/test-bot-claude-v1,